# LeetCode 1366 — Rank Teams by Votes bằng Python

Link đề: https://leetcode.com/problems/rank-teams-by-votes/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Sorting, Counting

Skeleton LeetCode:

```python
class Solution:
    def rankTeams(self, votes: list[str]) -> str:
        ...
```

## 1. Mục tiêu học

1. Biến luật xếp hạng bằng lời ("nhiều phiếu hạng 1 hơn; hòa thì xét hạng 2; …; cuối cùng theo chữ cái") thành **so sánh từ điển (lexicographic)** trên **vector đếm**.
2. Dựng **ma trận đếm** `cnt[team][pos]` = số cử tri xếp `team` ở vị trí `pos`.
3. Sắp xếp với **khóa nhiều tầng**: tuple `(−cnt[0], −cnt[1], …, tên)`; so sánh với `functools.cmp_to_key`.
4. Hiểu **sắp xếp ổn định (stable sort)** và mẹo "sắp theo tên trước, rồi sắp theo vector đếm giảm dần".
5. Tránh lỗi: chỉ đếm hạng 1, `reverse=True` làm đảo cả tiêu chí tên, dùng tổng điểm (Borda) thay vì so sánh từ điển.

## 2. Đề bài & ví dụ

Mỗi cử tri xếp hạng **tất cả** các đội từ cao xuống thấp (một chuỗi, ký tự đầu là hạng 1). Thứ tự các đội được quyết định như sau:
- Đội nào có **nhiều phiếu hạng 1** hơn đứng trước.
- Nếu hòa, xét **số phiếu hạng 2**; nếu vẫn hòa, xét hạng 3, … cho tới khi phân định.
- Nếu hòa ở **mọi** vị trí, xếp theo **chữ cái** của tên đội (tăng dần).

Cho mảng `votes`, trả về chuỗi các đội đã sắp xếp theo luật trên.

**Ví dụ 1:** `votes = ["ABC","ACB","ABC","ACB","ACB"]` → `"ACB"` (A có 5 phiếu hạng 1; C có 3 phiếu hạng 2, B có 2).

**Ví dụ 2:** `votes = ["WXYZ","XYZW"]` → `"XWYZ"` (W và X cùng 1 phiếu hạng 1; X có 1 phiếu hạng 2, W không có).

**Ví dụ 3:** `votes = ["ZMNAGUEDSJYLBOPHRQICWFXTVK"]` → giữ nguyên (chỉ một cử tri).

**Ràng buộc:** `1 <= votes.length <= 1000`; `1 <= votes[i].length <= 26`; mọi chuỗi cùng độ dài; chữ in hoa, không lặp trong một chuỗi; mọi chuỗi chứa cùng tập đội.

In [1]:
import random
import string
import time
from collections import defaultdict
from functools import cmp_to_key

examples = [
    (["ABC", "ACB", "ABC", "ACB", "ACB"], "ACB"),
    (["WXYZ", "XYZW"], "XWYZ"),
    (["ZMNAGUEDSJYLBOPHRQICWFXTVK"], "ZMNAGUEDSJYLBOPHRQICWFXTVK"),
]
for votes, want in examples:
    print(f"{len(votes)} phiếu, {len(votes[0])} đội -> mong đợi {want!r}")

5 phiếu, 3 đội -> mong đợi 'ACB'
2 phiếu, 4 đội -> mong đợi 'XWYZ'
1 phiếu, 26 đội -> mong đợi 'ZMNAGUEDSJYLBOPHRQICWFXTVK'


## 3. Kiến thức nền

### 3.1 So sánh tuple/list theo thứ tự từ điển

Python so sánh tuple **từng phần tử từ trái sang phải**, phần tử đầu tiên khác nhau quyết định. Đúng y luật "hạng 1 trước, hòa thì hạng 2…".

In [2]:
print((5, 0, 0) > (0, 3, 2))       # hơn ngay ở phần tử đầu
print((0, 3, 2) > (0, 2, 3))       # hòa phần tử đầu, hơn ở phần tử thứ hai
print((1, 0) == (1, 0), [2, 1] < [2, 3])

True
True
True True


### 3.2 Ma trận đếm bằng dict các list

`cnt[team]` là list dài `k` (số đội); `cnt[team][pos] += 1` mỗi khi một phiếu xếp `team` ở vị trí `pos`. `enumerate(vote)` cho cặp `(pos, team)`.

In [3]:
votes = ["ABC", "ACB", "ABC", "ACB", "ACB"]
k = len(votes[0])
cnt = {t: [0] * k for t in votes[0]}
for v in votes:
    for pos, t in enumerate(v):
        cnt[t][pos] += 1
print(cnt)

{'A': [5, 0, 0], 'B': [0, 2, 3], 'C': [0, 3, 2]}


### 3.3 `sorted` với `key` trả về tuple, và mẹo "đảo dấu"

Muốn **giảm dần** theo số đếm nhưng **tăng dần** theo tên: dùng khóa `(−c0, −c1, …, tên)` rồi sắp **tăng dần**. Toán tử `*` mở list vào tuple.

In [4]:
key = lambda t: (*[-c for c in cnt[t]], t)
print({t: key(t) for t in cnt})
print("".join(sorted(cnt, key=key)))

{'A': (-5, 0, 0, 'A'), 'B': (0, -2, -3, 'B'), 'C': (0, -3, -2, 'C')}
ACB


### 3.4 `functools.cmp_to_key` — viết hàm so sánh như C/Java

Hàm `cmp(a, b)` trả về số âm nếu `a` đứng trước `b`, dương nếu sau, 0 nếu bằng. `cmp_to_key(cmp)` biến nó thành `key` cho `sorted`.

In [5]:
def cmp_len_then_alpha(a, b):
    if len(a) != len(b):
        return len(a) - len(b)
    return -1 if a < b else (1 if a > b else 0)

print(sorted(["ccc", "b", "aa", "a"], key=cmp_to_key(cmp_len_then_alpha)))

['a', 'b', 'aa', 'ccc']


### 3.5 Sắp xếp ổn định (stable) và `reverse=True`

`sorted` của Python **ổn định**: phần tử có khóa bằng nhau giữ nguyên thứ tự ban đầu — kể cả khi `reverse=True` (Python đảm bảo điều này). Vì vậy có thể sắp **hai lượt**: theo tên tăng dần, rồi theo vector đếm giảm dần.

In [6]:
data = [("B", 2), ("A", 2), ("C", 5)]
step1 = sorted(data)                                    # theo tên
step2 = sorted(step1, key=lambda p: p[1], reverse=True)  # theo số giảm dần, ổn định
print(step1, "->", step2, "(A trước B vì hòa và A đứng trước sau lượt 1)")

[('A', 2), ('B', 2), ('C', 5)] -> [('C', 5), ('A', 2), ('B', 2)] (A trước B vì hòa và A đứng trước sau lượt 1)


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** `["ABC","ACB","ABC","ACB","ACB"]`:

| Đội | hạng 1 | hạng 2 | hạng 3 |
|---|---|---|---|
| A | 5 | 0 | 0 |
| B | 0 | 2 | 3 |
| C | 0 | 3 | 2 |

So sánh `(5,0,0) > (0,3,2) > (0,2,3)` ⇒ `A, C, B` ⇒ `"ACB"` ✔.

**Ví dụ 2** `["WXYZ","XYZW"]`: W `(1,0,0,1)`, X `(1,1,0,0)`, Y `(0,1,1,0)`, Z `(0,0,1,1)`. Thứ tự giảm dần: `X (1,1,0,0) > W (1,0,0,1) > Y > Z` ⇒ `"XWYZ"` ✔.

**Trực giác:** mỗi đội ↔ một vector đếm; luật đề = so sánh từ điển vector đó (giảm dần), hòa hoàn toàn thì theo tên.

In [7]:
def count_matrix(votes):
    k = len(votes[0])
    cnt = {t: [0] * k for t in votes[0]}
    for v in votes:
        for pos, t in enumerate(v):
            cnt[t][pos] += 1
    return cnt

for votes, want in examples[:2]:
    cnt = count_matrix(votes)
    for t in sorted(cnt):
        print(f"  {t}: {cnt[t]}")
    print("  ->", "".join(sorted(cnt, key=lambda t: (*[-c for c in cnt[t]], t))), "| mong đợi", want)

  A: [5, 0, 0]
  B: [0, 2, 3]
  C: [0, 3, 2]
  -> ACB | mong đợi ACB
  W: [1, 0, 0, 1]
  X: [1, 1, 0, 0]
  Y: [0, 1, 1, 0]
  Z: [0, 0, 1, 1]
  -> XWYZ | mong đợi XWYZ


## 5. Cách 1 — Brute Force / baseline: so sánh trực tiếp từ phiếu bầu

**Ý tưởng:** không dựng ma trận. Viết hàm so sánh `cmp(a, b)`: với từng vị trí `pos`, **đếm lại** trong toàn bộ `votes` số phiếu xếp `a` và `b` ở `pos`; vị trí đầu tiên khác nhau quyết định. Hòa hết thì so tên. Dùng `cmp_to_key` để sắp.

In [8]:
def rank_brute(votes):
    k = len(votes[0])

    def cmp(a, b):
        for pos in range(k):
            ca = sum(v[pos] == a for v in votes)       # đếm lại mỗi lần so sánh
            cb = sum(v[pos] == b for v in votes)
            if ca != cb:
                return cb - ca                          # nhiều phiếu hơn đứng trước
        return -1 if a < b else (1 if a > b else 0)

    return "".join(sorted(votes[0], key=cmp_to_key(cmp)))

for votes, want in examples:
    print(f"-> {rank_brute(votes)!r} (mong đợi {want!r})")

-> 'ACB' (mong đợi 'ACB')
-> 'XWYZ' (mong đợi 'XWYZ')
-> 'ZMNAGUEDSJYLBOPHRQICWFXTVK' (mong đợi 'ZMNAGUEDSJYLBOPHRQICWFXTVK')


**Độ phức tạp:** sắp `k` đội cần `O(k log k)` lần so sánh, mỗi lần tối đa `k` vị trí × `m` phiếu ⇒ `O(k² log k · m)` ≈ `26² × 5 × 1000 ≈ 3,4·10^6` — chạy được, nhưng lãng phí vì **đếm lặp lại**.

**Nút thắt:** số phiếu của một đội ở một vị trí là **cố định** — đếm **một lần** cho tất cả, lưu thành ma trận.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** tiền xử lý ma trận `cnt[team][pos]` trong `O(m·k)`. Khi đó luật xếp hạng là:

> `a` đứng trước `b` ⇔ `cnt[a]` **lớn hơn** `cnt[b]` theo thứ tự từ điển, hoặc hai vector bằng nhau và `a < b`.

**Quy tắc nếu… thì…:**
- Nếu muốn dùng `key`: khóa `(−cnt[t][0], …, −cnt[t][k−1], t)`, sắp tăng dần.
- Nếu muốn dùng `cmp`: so sánh `cnt[b]` với `cnt[a]` trước, rồi `a` với `b`.
- Nếu muốn dùng tính ổn định: sắp theo tên, rồi sắp theo `cnt[t]` với `reverse=True`.

**Bất biến:** sau khi đếm xong, `Σ_team cnt[team][pos] = m` cho mọi `pos` và `Σ_pos cnt[team][pos] = m` cho mọi `team`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Ma trận đếm + khóa tuple đảo dấu

**Giải thích từng đoạn:**
1. `cnt = {t: [0]*k for t in votes[0]}` — chỉ các đội có mặt.
2. Duyệt mọi phiếu, `cnt[t][pos] += 1`.
3. `sorted(teams, key=lambda t: (*(-c for c in cnt[t]), t))`.

In [9]:
def rank_key(votes):
    k = len(votes[0])
    cnt = {t: [0] * k for t in votes[0]}
    for v in votes:
        for pos, t in enumerate(v):
            cnt[t][pos] += 1
    return "".join(sorted(cnt, key=lambda t: (*[-c for c in cnt[t]], t)))

for votes, want in examples:
    print(f"-> {rank_key(votes)!r} (mong đợi {want!r})")

-> 'ACB' (mong đợi 'ACB')
-> 'XWYZ' (mong đợi 'XWYZ')
-> 'ZMNAGUEDSJYLBOPHRQICWFXTVK' (mong đợi 'ZMNAGUEDSJYLBOPHRQICWFXTVK')


Code nộp LeetCode (Cách 2):

In [10]:
class Solution:
    def rankTeams(self, votes: list[str]) -> str:
        k = len(votes[0])
        cnt = {t: [0] * k for t in votes[0]}
        for v in votes:
            for pos, t in enumerate(v):
                cnt[t][pos] += 1
        return "".join(sorted(cnt, key=lambda t: (*[-c for c in cnt[t]], t)))

print(Solution().rankTeams(["WXYZ", "XYZW"]))

XWYZ


**Độ phức tạp:** đếm `O(m·k)`, sắp `O(k log k)` lần so sánh tuple dài `k+1` ⇒ `O(m·k + k² log k)`. Bộ nhớ `O(k²)`.

### 7.2 Cách 3 — Ma trận đếm + `cmp_to_key`

**Giải thích:** hàm so sánh dùng ma trận đã đếm; so sánh list trực tiếp (`cnt[a] > cnt[b]` là so sánh từ điển). Cách này "đọc như đề bài" — thích hợp khi luật phức tạp hơn khó gói vào một tuple.

In [11]:
def rank_cmp(votes):
    cnt = count_matrix(votes)

    def cmp(a, b):
        if cnt[a] != cnt[b]:
            return -1 if cnt[a] > cnt[b] else 1      # vector lớn hơn đứng trước
        return -1 if a < b else (1 if a > b else 0)

    return "".join(sorted(cnt, key=cmp_to_key(cmp)))

for votes, want in examples:
    print(f"-> {rank_cmp(votes)!r} (mong đợi {want!r})")

-> 'ACB' (mong đợi 'ACB')
-> 'XWYZ' (mong đợi 'XWYZ')
-> 'ZMNAGUEDSJYLBOPHRQICWFXTVK' (mong đợi 'ZMNAGUEDSJYLBOPHRQICWFXTVK')


Code nộp LeetCode (Cách 3):

In [12]:
class Solution:
    def rankTeams(self, votes: list[str]) -> str:
        k = len(votes[0])
        cnt = {t: [0] * k for t in votes[0]}
        for v in votes:
            for pos, t in enumerate(v):
                cnt[t][pos] += 1

        def cmp(a, b):
            if cnt[a] != cnt[b]:
                return -1 if cnt[a] > cnt[b] else 1
            return -1 if a < b else (1 if a > b else 0)

        return "".join(sorted(cnt, key=cmp_to_key(cmp)))

print(Solution().rankTeams(["ABC", "ACB", "ABC", "ACB", "ACB"]))

ACB


### 7.3 Cách 4 — Hai lượt sắp xếp ổn định

**Giải thích:** (1) sắp các đội theo **tên** tăng dần; (2) sắp lại theo **vector đếm** với `reverse=True`. Nhờ tính ổn định, các đội có vector bằng nhau giữ thứ tự chữ cái từ lượt 1. Không cần đảo dấu.

In [13]:
def rank_stable(votes):
    cnt = count_matrix(votes)
    teams = sorted(cnt)                                     # lượt 1: theo tên
    teams.sort(key=lambda t: cnt[t], reverse=True)          # lượt 2: theo vector, ổn định
    return "".join(teams)

for votes, want in examples:
    print(f"-> {rank_stable(votes)!r} (mong đợi {want!r})")

-> 'ACB' (mong đợi 'ACB')
-> 'XWYZ' (mong đợi 'XWYZ')
-> 'ZMNAGUEDSJYLBOPHRQICWFXTVK' (mong đợi 'ZMNAGUEDSJYLBOPHRQICWFXTVK')


### 7.4 Cách 5 — Mã hóa thành một số nguyên duy nhất

**Giải thích:** vì mỗi `cnt ≤ 1000 < 1001`, ta mã hóa vector thành số hệ cơ số `B = 1001`: `code = Σ cnt[pos] · B^(k−1−pos)`. So sánh số ⇔ so sánh từ điển vector. Khóa `(−code, tên)`. Minh họa kỹ thuật "nén nhiều tiêu chí thành một số" (dùng nhiều trong SQL/ngôn ngữ không có tuple).

In [14]:
def rank_encode(votes):
    cnt = count_matrix(votes)
    B = len(votes) + 1                       # mỗi chữ số < B
    def encode(t):
        code = 0
        for c in cnt[t]:
            code = code * B + c
        return code
    return "".join(sorted(cnt, key=lambda t: (-encode(t), t)))

for votes, want in examples:
    print(f"-> {rank_encode(votes)!r} (mong đợi {want!r})")

-> 'ACB' (mong đợi 'ACB')
-> 'XWYZ' (mong đợi 'XWYZ')
-> 'ZMNAGUEDSJYLBOPHRQICWFXTVK' (mong đợi 'ZMNAGUEDSJYLBOPHRQICWFXTVK')


## 8. TRACE từng bước

Trace Cách 2 trên Ví dụ 2 `["WXYZ","XYZW"]`: in ma trận sau mỗi phiếu, rồi khóa sắp xếp.

| Sau phiếu | W | X | Y | Z |
|---|---|---|---|---|
| WXYZ | [1,0,0,0] | [0,1,0,0] | [0,0,1,0] | [0,0,0,1] |
| XYZW | [1,0,0,1] | [1,1,0,0] | [0,1,1,0] | [0,0,1,1] |

Khóa: X `(−1,−1,0,0,'X')` < W `(−1,0,0,−1,'W')` < Y `(0,−1,−1,0,'Y')` < Z `(0,0,−1,−1,'Z')` ⇒ `"XWYZ"`.

In [15]:
def trace_rank(votes):
    k = len(votes[0])
    cnt = {t: [0] * k for t in votes[0]}
    for v in votes:
        for pos, t in enumerate(v):
            cnt[t][pos] += 1
        print(f"sau phiếu {v!r}: " + ", ".join(f"{t}={cnt[t]}" for t in sorted(cnt)))
    keys = {t: (*[-c for c in cnt[t]], t) for t in cnt}
    order = sorted(cnt, key=lambda t: keys[t])
    for t in order:
        print(f"  khóa {t}: {keys[t]}")
    print("=> kết quả:", "".join(order))
    return "".join(order)

trace_rank(["WXYZ", "XYZW"])
print()
trace_rank(["ABC", "ACB", "ABC", "ACB", "ACB"])

sau phiếu 'WXYZ': W=[1, 0, 0, 0], X=[0, 1, 0, 0], Y=[0, 0, 1, 0], Z=[0, 0, 0, 1]
sau phiếu 'XYZW': W=[1, 0, 0, 1], X=[1, 1, 0, 0], Y=[0, 1, 1, 0], Z=[0, 0, 1, 1]
  khóa X: (-1, -1, 0, 0, 'X')
  khóa W: (-1, 0, 0, -1, 'W')
  khóa Y: (0, -1, -1, 0, 'Y')
  khóa Z: (0, 0, -1, -1, 'Z')
=> kết quả: XWYZ

sau phiếu 'ABC': A=[1, 0, 0], B=[0, 1, 0], C=[0, 0, 1]
sau phiếu 'ACB': A=[2, 0, 0], B=[0, 1, 1], C=[0, 1, 1]
sau phiếu 'ABC': A=[3, 0, 0], B=[0, 2, 1], C=[0, 1, 2]
sau phiếu 'ACB': A=[4, 0, 0], B=[0, 2, 2], C=[0, 2, 2]
sau phiếu 'ACB': A=[5, 0, 0], B=[0, 2, 3], C=[0, 3, 2]
  khóa A: (-5, 0, 0, 'A')
  khóa C: (0, -3, -2, 'C')
  khóa B: (0, -2, -3, 'B')
=> kết quả: ACB


'ACB'

Trace Cách 4 (hai lượt ổn định) trên một ví dụ có hòa hoàn toàn: `["BCA","CAB","ABC"]` (mỗi đội đúng một phiếu ở mỗi vị trí).

In [16]:
def trace_stable(votes):
    cnt = count_matrix(votes)
    teams = sorted(cnt)
    print("lượt 1 (theo tên):", teams, {t: cnt[t] for t in teams})
    teams.sort(key=lambda t: cnt[t], reverse=True)
    print("lượt 2 (theo vector giảm, ổn định):", teams)
    return "".join(teams)

print("=>", trace_stable(["BCA", "CAB", "ABC"]))

lượt 1 (theo tên): ['A', 'B', 'C'] {'A': [1, 1, 1], 'B': [1, 1, 1], 'C': [1, 1, 1]}
lượt 2 (theo vector giảm, ổn định): ['A', 'B', 'C']
=> ABC


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def rankTeams(self, votes: list[str]) -> str:
        k = len(votes[0])
        cnt = {t: [0] * k for t in votes[0]}          # cnt[đội][vị trí]
        for v in votes:
            for pos, t in enumerate(v):
                cnt[t][pos] += 1
        # nhiều phiếu ở vị trí sớm hơn -> đứng trước; hòa hết -> theo chữ cái
        return "".join(sorted(cnt, key=lambda t: (*[-c for c in cnt[t]], t)))

sol = Solution()
for votes, want in examples:
    assert sol.rankTeams(votes) == want
    print(f"rankTeams({votes}) = {sol.rankTeams(votes)!r}")

rankTeams(['ABC', 'ACB', 'ABC', 'ACB', 'ACB']) = 'ACB'
rankTeams(['WXYZ', 'XYZW']) = 'XWYZ'
rankTeams(['ZMNAGUEDSJYLBOPHRQICWFXTVK']) = 'ZMNAGUEDSJYLBOPHRQICWFXTVK'


## 10. Vì sao thuật toán đúng

1. **Ma trận đếm đúng:** mỗi phiếu đóng góp đúng `+1` vào `cnt[t][pos]` cho mỗi cặp (đội, vị trí) xuất hiện trong phiếu ⇒ `cnt[t][pos]` = số cử tri xếp `t` ở vị trí `pos`.
2. **Khóa mã hóa đúng luật:** với hai đội `a ≠ b`, so sánh `key(a) = (−cnt[a][0], …, a)` và `key(b)`: phần tử đầu tiên khác nhau là vị trí `pos` nhỏ nhất mà `cnt[a][pos] ≠ cnt[b][pos]` — đúng vị trí đề yêu cầu xét; `−cnt[a][pos] < −cnt[b][pos]` ⇔ `a` có nhiều phiếu hơn ⇔ `a` đứng trước. Nếu không có vị trí nào khác nhau, phần tử cuối (tên) quyết định theo chữ cái tăng dần ✔.
3. **Thứ tự toàn phần:** khóa là duy nhất cho mỗi đội (tên khác nhau) ⇒ sắp xếp cho kết quả xác định.
4. **Cách 4:** lượt 2 sắp theo vector giảm dần; với các đội có vector bằng nhau, tính ổn định giữ thứ tự của lượt 1 (chữ cái) ⇒ tương đương khóa ở bước 2.
5. **Cách 5:** mọi `cnt[t][pos] ≤ m < B` nên biểu diễn cơ số `B` là song ánh và bảo toàn thứ tự từ điển.

## 11. Phân tích Time / Space

Gọi `m` = số phiếu (≤ 1000), `k` = số đội (≤ 26).

| Cách | Time | Space |
|---|---|---|
| 1. So sánh bằng cách đếm lại | `O(k log k · k · m)` | `O(k)` |
| 2. Ma trận + khóa tuple | `O(m·k + k² log k)` | `O(k²)` |
| 3. Ma trận + cmp_to_key | `O(m·k + k² log k)` | `O(k²)` |
| 4. Hai lượt ổn định | `O(m·k + k² log k)` | `O(k²)` |
| 5. Mã hóa số nguyên | `O(m·k + k² + k log k)` | `O(k²)` |

Phần chi phối là đếm `O(m·k) ≤ 26 000` — rất nhỏ.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
METHODS = {
    "brute": rank_brute,
    "key": rank_key,
    "cmp": rank_cmp,
    "stable": rank_stable,
    "encode": rank_encode,
    "submit": Solution().rankTeams,
}
cases = examples + [
    (["M"], "M"),                                              # 1 đội
    (["M", "M", "M", "M"], "M"),
    (["BCA", "CAB", "ABC"], "ABC"),                            # hòa hoàn toàn -> chữ cái
    (["BA", "AB"], "AB"),                                      # hòa -> chữ cái
    (["BCA", "CAB", "CBA", "ABC", "ACB", "BAC"], "ABC"),       # hòa hoàn toàn
    (["ZYX", "ZXY", "YZX"], "ZYX"),
    (["CBA", "CBA", "ABC"], "CAB"),                          # C[2,0,1] > A[1,0,2] > B[0,3,0]
]
for votes, want in cases:
    for name, f in METHODS.items():
        got = f(votes)
        assert got == want, (name, votes, got, want)
print(f"{len(cases)} ca × {len(METHODS)} cách: đúng hết")

10 ca × 6 cách: đúng hết


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Sinh `k` đội ngẫu nhiên (tập con của A–Z), `m` phiếu là các hoán vị ngẫu nhiên. Để có nhiều trường hợp hòa, dùng `k` nhỏ và `m` nhỏ. Kiểm tra thêm tính chất: kết quả là một hoán vị của các đội.

In [19]:
rng = random.Random(1366)
for _ in range(500):
    k = rng.randint(1, 6)
    teams = rng.sample(string.ascii_uppercase, k)
    m = rng.randint(1, 8)
    votes = ["".join(rng.sample(teams, k)) for _ in range(m)]
    ref = rank_brute(votes)
    assert sorted(ref) == sorted(teams)
    for name, f in METHODS.items():
        assert f(votes) == ref, (name, votes)
print("500 bộ phiếu ngẫu nhiên: mọi cách khớp brute force")

500 bộ phiếu ngẫu nhiên: mọi cách khớp brute force


### 12.3 Kích thước lớn nhất: 1000 phiếu × 26 đội

In [20]:
rng = random.Random(9)
teams = list(string.ascii_uppercase)
votes = ["".join(rng.sample(teams, 26)) for _ in range(1000)]
res = {}
for name, f in METHODS.items():
    st = time.perf_counter()
    res[name] = f(votes)
    print(f"{name:7} -> {res[name]}  ({time.perf_counter() - st:.4f}s)")
assert len(set(res.values())) == 1

brute   -> JOMDRLICHSFYKEXGQUVNWBZPAT  (0.0068s)
key     -> JOMDRLICHSFYKEXGQUVNWBZPAT  (0.0014s)
cmp     -> JOMDRLICHSFYKEXGQUVNWBZPAT  (0.0013s)
stable  -> JOMDRLICHSFYKEXGQUVNWBZPAT  (0.0011s)
encode  -> JOMDRLICHSFYKEXGQUVNWBZPAT  (0.0012s)
submit  -> JOMDRLICHSFYKEXGQUVNWBZPAT  (0.0014s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ đếm phiếu hạng 1

In [21]:
def wrong_first_only(votes):
    first = defaultdict(int)
    for v in votes:
        first[v[0]] += 1
    return "".join(sorted(votes[0], key=lambda t: (-first[t], t)))

for votes, want in examples[:2]:
    print(f"sai cho {wrong_first_only(votes)!r}, đúng là {want!r}")

sai cho 'ABC', đúng là 'ACB'
sai cho 'WXYZ', đúng là 'XWYZ'


**Vì sao sai:** khi hòa hạng 1 (hoặc nhiều đội 0 phiếu hạng 1) phải xét tiếp hạng 2, 3… chứ không nhảy ngay sang chữ cái. **Sửa:** đếm **mọi** vị trí.

### Lỗi 2 — `reverse=True` với khóa `(vector, tên)` làm đảo cả tên

In [22]:
def wrong_reverse_name(votes):
    cnt = count_matrix(votes)
    return "".join(sorted(cnt, key=lambda t: (cnt[t], t), reverse=True))

for votes in (["BCA", "CAB", "ABC"], ["BA", "AB"]):
    print(f"{votes}: sai cho {wrong_reverse_name(votes)!r}, đúng là {rank_key(votes)!r}")

['BCA', 'CAB', 'ABC']: sai cho 'CBA', đúng là 'ABC'
['BA', 'AB']: sai cho 'BA', đúng là 'AB'


**Vì sao sai:** `reverse=True` đảo **toàn bộ** khóa, nên khi hòa vector, tên bị xếp Z→A. **Sửa:** đảo dấu riêng phần số `(−c…, tên)`, hoặc dùng hai lượt ổn định (Cách 4).

### Lỗi 3 — Không có tiêu chí phụ theo tên (dựa vào thứ tự xuất hiện)

In [23]:
def wrong_no_tiebreak(votes):
    cnt = count_matrix(votes)
    return "".join(sorted(votes[0], key=lambda t: cnt[t], reverse=True))   # hòa giữ thứ tự votes[0]

for votes in (["BCA", "CAB", "ABC"], ["BA", "AB"]):
    print(f"{votes}: sai cho {wrong_no_tiebreak(votes)!r}, đúng là {rank_key(votes)!r}")

['BCA', 'CAB', 'ABC']: sai cho 'BCA', đúng là 'ABC'
['BA', 'AB']: sai cho 'BA', đúng là 'AB'


**Vì sao sai:** hòa hoàn toàn thì kết quả theo thứ tự trong `votes[0]` (ở đây `"BCA"`), không theo chữ cái. **Sửa:** sắp theo tên **trước** rồi mới sắp theo vector (Cách 4), hoặc đưa tên vào khóa.

### Lỗi 4 — Dùng tổng điểm (Borda count) thay vì so sánh từ điển

In [24]:
def wrong_borda(votes):
    k = len(votes[0])
    score = defaultdict(int)
    for v in votes:
        for pos, t in enumerate(v):
            score[t] += k - pos                    # hạng 1 được k điểm, ...
    return "".join(sorted(votes[0], key=lambda t: (-score[t], t)))

votes = ["ACB", "ACB", "BCA", "BCA", "CBA"]
print("ma trận:", count_matrix(votes))
print(f"sai cho {wrong_borda(votes)!r}, đúng là {rank_key(votes)!r}")

ma trận: {'A': [2, 0, 3], 'C': [1, 4, 0], 'B': [2, 1, 2]}
sai cho 'CBA', đúng là 'BAC'


**Vì sao sai:** đề xét **lần lượt** từng vị trí, không cộng gộp. Ở ví dụ, A `[2,0,3]` và B `[2,1,2]` cùng 2 phiếu hạng 1, B hơn ở hạng 2 ⇒ B trước A; C `[1,4,0]` chỉ có 1 phiếu hạng 1 nên đứng **cuối** — dù tổng điểm Borda của C cao nhất (11 so với 10 và 9). **Sửa:** so sánh vector theo thứ tự từ điển.

### Lỗi 5 — Dựng ma trận kích thước 26×26 rồi xuất cả các đội không tham gia

In [25]:
def wrong_all_letters(votes):
    k = len(votes[0])
    cnt = {t: [0] * k for t in string.ascii_uppercase}   # mọi chữ cái
    for v in votes:
        for pos, t in enumerate(v):
            cnt[t][pos] += 1
    return "".join(sorted(cnt, key=lambda t: (*[-c for c in cnt[t]], t)))

print(f"sai cho {wrong_all_letters(['WXYZ', 'XYZW'])!r}, đúng là 'XWYZ'")

sai cho 'XWYZABCDEFGHIJKLMNOPQRSTUV', đúng là 'XWYZ'


**Vì sao sai:** các chữ cái không có trong phiếu (vector toàn 0) bị nối vào kết quả. **Sửa:** chỉ xếp các đội trong `votes[0]` (hoặc lọc đội có tổng phiếu > 0).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Ưu tiên tiêu chí 1, hòa thì tiêu chí 2, …"** ⇒ khóa tuple; Python so sánh tuple đúng kiểu từ điển.
- Giảm dần cho số + tăng dần cho chuỗi ⇒ **đảo dấu số**, không dùng `reverse=True` cho cả khóa.
- Luật quá phức tạp cho tuple ⇒ `cmp_to_key`.
- Tính **ổn định** của `sort` cho phép sắp nhiều lượt từ tiêu chí phụ tới tiêu chí chính.
- Bảng "đối tượng × vị trí" ⇒ ma trận đếm tiền xử lý một lần.

## 15. Biến thể của bài

1. **Phiếu có trọng số** (mỗi cử tri một hệ số) — cộng trọng số thay vì `+1` (Bài 5).
2. **Phiếu không đầy đủ** (cử tri chỉ xếp vài đội đầu) — ma trận vẫn đúng, đội vắng chỉ không nhận phiếu ở vị trí đó; cần tập đội là hợp của mọi phiếu.
3. **Borda count** (cộng điểm) — biến thể "khác luật" ở Lỗi 4; dùng một số nguyên làm khóa.
4. **LeetCode 1636 — Sort Array by Increasing Frequency**, **451 — Sort Characters By Frequency**: cùng kỹ thuật khóa nhiều tầng.
5. **Trả về thứ hạng của một đội cụ thể** mà không sắp toàn bộ — đếm số đội có khóa nhỏ hơn.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So sánh trực tiếp | đếm lại khi so sánh | `O(k² m log k)` | `O(k)` | baseline, trọng tài |
| 2. Ma trận + khóa tuple | `(−c0, −c1, …, tên)` | `O(mk + k² log k)` | `O(k²)` | **nộp bài**, ngắn gọn |
| 3. Ma trận + cmp_to_key | hàm so sánh như đề | `O(mk + k² log k)` | `O(k²)` | luật phức tạp |
| 4. Hai lượt ổn định | tên trước, vector sau | `O(mk + k² log k)` | `O(k²)` | tránh đảo dấu |
| 5. Mã hóa số | cơ số `m+1` | `O(mk + k²)` | `O(k²)` | khi chỉ được dùng khóa số |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `rankTeams(["BCA","CAB","CBA","ABC","ACB","BAC"])` và `rankTeams(["FVSHJIEMNGYPTQOURLWCZKAX","AITFQORCEHPVJMXGKSLNZWUY"])` (chỉ cần đoán hai ký tự đầu của kết quả thứ hai).

**Bài 2 (tính tay):** Lập ma trận đếm và kết quả cho `["ABC","BAC","BCA","CAB"]`.

**Bài 3 (điền TODO):** Hoàn thiện `build_matrix(votes)` trả về dict `đội → list đếm`.

**Bài 4 (tự viết lại):** Viết lại lời giải chỉ dùng **list 26×k** (chỉ số `ord(t) − 65`) và mẹo hai lượt ổn định.

**Bài 5 (mini challenge):** `rank_weighted(votes, weights)`: phiếu `votes[i]` có trọng số nguyên `weights[i] ≥ 1`; luật như cũ nhưng "số phiếu" thay bằng **tổng trọng số**.

In [26]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def build_matrix(votes):
    k = len(votes[0])
    cnt = {}
    # TODO: khởi tạo cnt[t] = [0] * k cho mỗi đội t trong votes[0]
    # TODO: với mỗi phiếu v, với mỗi (pos, t) trong enumerate(v): cnt[t][pos] += 1
    return cnt

In [27]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def rank_array26(votes):
    # TODO: rows = [[0] * k for _ in range(26)]
    pass

def rank_weighted(votes, weights):
    # TODO: cnt[t][pos] += w
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Ca thứ nhất: mỗi đội có đúng 2 phiếu ở mỗi vị trí. (2) Hòa hoàn toàn ⇒ theo chữ cái. (3) Ca thứ hai: hai đội có phiếu hạng 1 là F và A, mỗi đội 1 phiếu — xét hạng 2: V (của F) và I (của A) không phải F/A, nên F và A đều 0 phiếu hạng 2… tiếp tục xét đến khi phân định, hoặc cứ chạy code để kiểm tra.

**Bài 2:** (1) Mỗi phiếu dài 3 ⇒ vector dài 3, tổng mỗi vector = 4. (2) A `[1,2,1]`, B `[2,1,1]`, C `[1,1,2]`. (3) B có nhiều phiếu hạng 1 nhất; A và C hòa hạng 1, A hơn ở hạng 2.

**Bài 3:** (1) Dict comprehension. (2) Hai vòng lặp lồng. (3) Kiểm tra tổng mỗi cột = số phiếu.

**Bài 4:** (1) `rows[ord(t) − 65][pos] += 1`. (2) `teams = sorted(votes[0])`. (3) `teams.sort(key=lambda t: rows[ord(t) − 65], reverse=True)`.

**Bài 5:** (1) `zip(votes, weights)`. (2) Cộng `w` thay vì 1. (3) Kiểm chứng: trọng số `w` tương đương nhân bản phiếu đó `w` lần rồi dùng `rankTeams`.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
r1 = rank_key(["BCA", "CAB", "CBA", "ABC", "ACB", "BAC"])
r2 = rank_key(["FVSHJIEMNGYPTQOURLWCZKAX", "AITFQORCEHPVJMXGKSLNZWUY"])
print("Bài 1:", r1, r2)
assert r1 == "ABC" and r2 == rank_brute(["FVSHJIEMNGYPTQOURLWCZKAX", "AITFQORCEHPVJMXGKSLNZWUY"])

# Bài 2
cnt = count_matrix(["ABC", "BAC", "BCA", "CAB"])
print("Bài 2:", cnt, "->", rank_key(["ABC", "BAC", "BCA", "CAB"]))
assert cnt == {"A": [1, 2, 1], "B": [2, 1, 1], "C": [1, 1, 2]}
assert rank_key(["ABC", "BAC", "BCA", "CAB"]) == "BAC"

Bài 1: ABC FAIVTSHJQOERMCNGPYXUKLWZ
Bài 2: {'A': [1, 2, 1], 'B': [2, 1, 1], 'C': [1, 1, 2]} -> BAC


In [29]:
# Bài 3
def build_matrix_sol(votes):
    k = len(votes[0])
    cnt = {t: [0] * k for t in votes[0]}
    for v in votes:
        for pos, t in enumerate(v):
            cnt[t][pos] += 1
    return cnt

M = build_matrix_sol(["ABC", "ACB", "ABC", "ACB", "ACB"])
assert M == {"A": [5, 0, 0], "B": [0, 2, 3], "C": [0, 3, 2]}
assert all(sum(M[t][pos] for t in M) == 5 for pos in range(3))

# Bài 4
def rank_array26_sol(votes):
    k = len(votes[0])
    rows = [[0] * k for _ in range(26)]
    for v in votes:
        for pos, t in enumerate(v):
            rows[ord(t) - 65][pos] += 1
    teams = sorted(votes[0])
    teams.sort(key=lambda t: rows[ord(t) - 65], reverse=True)
    return "".join(teams)

rng = random.Random(4)
for _ in range(300):
    k = rng.randint(1, 7)
    teams = rng.sample(string.ascii_uppercase, k)
    votes = ["".join(rng.sample(teams, k)) for _ in range(rng.randint(1, 9))]
    assert rank_array26_sol(votes) == rank_brute(votes)
print("Bài 3, 4: đúng (Bài 4 khớp brute force trên 300 bộ)")

Bài 3, 4: đúng (Bài 4 khớp brute force trên 300 bộ)


In [30]:
# Bài 5
def rank_weighted_sol(votes, weights):
    k = len(votes[0])
    cnt = {t: [0] * k for t in votes[0]}
    for v, w in zip(votes, weights):
        for pos, t in enumerate(v):
            cnt[t][pos] += w
    return "".join(sorted(cnt, key=lambda t: (*[-c for c in cnt[t]], t)))

print("Bài 5:", rank_weighted_sol(["ABC", "CBA"], [1, 3]), rank_weighted_sol(["ABC", "CBA"], [2, 2]))
assert rank_weighted_sol(["ABC", "CBA"], [1, 3]) == "CAB"     # C [3,0,1] > A [1,0,3] > B [0,4,0]
assert rank_weighted_sol(["ABC", "CBA"], [2, 2]) == "ACB"     # A, C hòa [2,0,2] -> chữ cái; B [0,4,0] cuối
rng = random.Random(5)
for _ in range(300):
    k = rng.randint(1, 6)
    teams = rng.sample(string.ascii_uppercase, k)
    m = rng.randint(1, 6)
    votes = ["".join(rng.sample(teams, k)) for _ in range(m)]
    weights = [rng.randint(1, 4) for _ in range(m)]
    expanded = [v for v, w in zip(votes, weights) for _ in range(w)]   # nhân bản phiếu
    assert rank_weighted_sol(votes, weights) == rank_brute(expanded)
print("Bài 5: khớp cách 'nhân bản phiếu' + brute force trên 300 bộ")

Bài 5: CAB ACB


Bài 5: khớp cách 'nhân bản phiếu' + brute force trên 300 bộ


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
k = len(votes[0])
cnt = {t: [0] * k for t in votes[0]}
for v in votes:
    for pos, t in enumerate(v):
        cnt[t][pos] += 1
return "".join(sorted(cnt, key=lambda t: (*[-c for c in cnt[t]], t)))
```

Hoặc hai lượt ổn định: `teams = sorted(cnt); teams.sort(key=lambda t: cnt[t], reverse=True)`.

**Checklist trước khi submit:**
- [ ] Đếm **mọi** vị trí, không chỉ hạng 1.
- [ ] So sánh **từ điển** trên vector, không cộng điểm.
- [ ] Số giảm dần, tên tăng dần (đảo dấu số, không `reverse` cả khóa).
- [ ] Chỉ xuất các đội có trong phiếu.
- [ ] Thử ca hòa hoàn toàn `["BCA","CAB","ABC"]` ⇒ `"ABC"`.
- [ ] Đúng chữ ký: `rankTeams(self, votes: list[str]) -> str`.